# 05 — Skalierung & 3B-Kontrast (Phase 6)

Auswertung der finalen Pipeline (Iteration B) auf dem **vollen Korpus**. Die Predictions entstehen in `IterationB.ipynb`:
- `daten/predictions_7b_full.jsonl` (7B, finale Pipeline)
- `daten/predictions_3b_full.jsonl` (3B-Kontrast, Qwen2.5-3B-Instruct)

Eigenes Notebook, weil `03_eval.ipynb` (Phasen 3–4) schon sehr groß ist. Alle Zellen sind abgesichert: solange die `*_full.jsonl` fehlen, kommt nur ein Hinweis statt eines Fehlers.

In [1]:
from pathlib import Path
import json
import math

import pandas as pd

GOLD_PATH    = Path("../annotation/meine_gold.csv")
PRED_7B_FULL = Path("../daten/predictions_7b_full.jsonl")
PRED_3B_FULL = Path("../daten/predictions_3b_full.jsonl")

FIELDS = ["homeoffice", "vertragsart", "erfahrungslevel",
          "gehalt_min_eur", "gehalt_zeitraum", "skills_top3"]

# Hand-Gold laden (id -> refnr), Werte trimmen
gold = pd.read_csv(GOLD_PATH)
if "id" in gold.columns and "refnr" not in gold.columns:
    gold = gold.rename(columns={"id": "refnr"})
for c in gold.columns:
    if gold[c].dtype == object:
        gold[c] = gold[c].astype(str).str.strip()
gold["refnr"] = gold["refnr"].astype(str).str.strip()


def is_missing(v):
    if v is None:
        return True
    if isinstance(v, float) and math.isnan(v):
        return True
    if isinstance(v, str) and v.strip().lower() in {"", "nan", "none", "null"}:
        return True
    return False


def norm_scalar(v):
    return None if is_missing(v) else str(v).strip().lower()


def norm_salary(v):
    if is_missing(v):
        return None
    try:
        return int(float(v))
    except (ValueError, TypeError):
        return str(v).strip()


def norm_skills(v):
    if is_missing(v):
        return set()
    items = v if isinstance(v, list) else str(v).split("|")
    return {str(i).strip().lower() for i in items if not is_missing(i)}


def field_equal(field, gv, pv):
    if field == "skills_top3":
        return norm_skills(gv) == norm_skills(pv)
    if field == "gehalt_min_eur":
        return norm_salary(gv) == norm_salary(pv)
    return norm_scalar(gv) == norm_scalar(pv)


def load_full(path):
    p = Path(path)
    if not p.exists():
        print("Noch keine Datei:", p)
        return None
    df = pd.read_json(p, lines=True)
    df["refnr"] = df["refnr"].astype(str).str.strip()
    if "parse_ok" not in df.columns:
        df["parse_ok"] = True
    print(p.name, "| Anzeigen:", len(df),
          "| Parse-Fails:", int((~df["parse_ok"].fillna(False)).sum()))
    return df


pred_7b_full = load_full(PRED_7B_FULL)
pred_3b_full = load_full(PRED_3B_FULL)

<jemalloc>: Unsupported system page size


predictions_7b_full.jsonl | Anzeigen: 49 | Parse-Fails: 0
predictions_3b_full.jsonl | Anzeigen: 49 | Parse-Fails: 0


## 7B & 3B auf den 12 Hand-Gold-Anzeigen

`eval_on_gold` merged die full-Predictions auf die 12 Gold-Anzeigen und rechnet Per-Field-Accuracy. Bleibt die finale Pipeline (7B) stabil zur Phase-3/4-Auswertung? Und wie schlägt sich 3B auf genau denselben 12?

In [2]:
def eval_on_gold(pred_df, label):
    m = gold.merge(pred_df, on="refnr", suffixes=("_gold", "_pred"))
    rows = []
    for field in FIELDS:
        flags = []
        for _, r in m.iterrows():
            ok = bool(r.get("parse_ok", True))
            flags.append(ok and field_equal(field, r[field + "_gold"], r[field + "_pred"]))
        rows.append({"feld": field, label: round(sum(flags) / len(flags), 3) if flags else 0.0})
    return pd.DataFrame(rows)


tab7 = tab3 = None
if pred_7b_full is not None:
    tab7 = eval_on_gold(pred_7b_full, "7b_full")
    print("7B-full auf den 12 Gold-Anzeigen:")
    display(tab7)
if pred_3b_full is not None:
    tab3 = eval_on_gold(pred_3b_full, "3b_full")
    print("3B-full auf den 12 Gold-Anzeigen:")
    display(tab3)
if tab7 is not None and tab3 is not None:
    print("Per-Field-Accuracy: 7B-full vs 3B-full (12 Gold)")
    display(tab7.merge(tab3, on="feld"))

7B-full auf den 12 Gold-Anzeigen:


,feld,7b_full
0,homeoffice,0.583
1,vertragsart,0.833
2,erfahrungslevel,0.667
3,gehalt_min_eur,0.917
4,gehalt_zeitraum,0.917
5,skills_top3,0.000


3B-full auf den 12 Gold-Anzeigen:


,feld,3b_full
0,homeoffice,0.333
1,vertragsart,0.750
2,erfahrungslevel,0.583
3,gehalt_min_eur,0.417
4,gehalt_zeitraum,0.417
5,skills_top3,0.000


Per-Field-Accuracy: 7B-full vs 3B-full (12 Gold)


,feld,7b_full,3b_full
0,homeoffice,0.583,0.333
1,vertragsart,0.833,0.750
2,erfahrungslevel,0.667,0.583
3,gehalt_min_eur,0.917,0.417
4,gehalt_zeitraum,0.917,0.417
5,skills_top3,0.000,0.000


## 3B vs. 7B über den vollen Korpus

Auf allen Anzeigen (auch ohne Gold) lässt sich 3B gegen 7B vergleichen. Wo 3B von 7B abweicht **und** (wo Gold vorhanden) 7B mit dem Gold übereinstimmt, ist es eine **wahrscheinliche 3B-Halluzination** — die Materialbasis für die drei Klassen.

In [3]:
if pred_7b_full is not None and pred_3b_full is not None:
    keep = ["refnr"] + FIELDS
    m37 = pred_7b_full[keep].merge(pred_3b_full[keep], on="refnr", suffixes=("_7b", "_3b"))
    print("Gemeinsame Anzeigen 7B/3B:", len(m37))

    overview = []
    for field in FIELDS:
        s = [field_equal(field, a, b) for a, b in zip(m37[field + "_7b"], m37[field + "_3b"])]
        overview.append({"feld": field, "3b_gleich_7b": f"{sum(s)}/{len(s)}",
                         "abweichungen": len(s) - sum(s)})
    print("3B-Uebereinstimmung mit 7B pro Feld:")
    display(pd.DataFrame(overview))

    gold_ids = set(gold["refnr"])
    recs = []
    for _, r in m37.iterrows():
        rid = r["refnr"]
        has_gold = rid in gold_ids
        for field in FIELDS:
            v7, v3 = r[field + "_7b"], r[field + "_3b"]
            if field_equal(field, v7, v3):
                continue
            rec = {"refnr": rid, "feld": field, "wert_7b": v7, "wert_3b": v3,
                   "hat_gold": has_gold, "wert_gold": "", "3b_halluzination": ""}
            if has_gold:
                gv = gold.loc[gold["refnr"] == rid, field].iloc[0]
                rec["wert_gold"] = gv
                rec["3b_halluzination"] = field_equal(field, v7, gv) and not field_equal(field, v3, gv)
            recs.append(rec)

    div = pd.DataFrame(recs)
    print("Abweichungen 3B vs. 7B gesamt:", len(div))
    display(div)

    if not div.empty:
        hall = div[div["3b_halluzination"] == True]
        print("Wahrscheinliche 3B-Halluzinationen (7B == Gold, 3B != Gold):")
        display(hall if not hall.empty else "keine auf den 12 Gold-Anzeigen")
else:
    print("Es fehlt mindestens eine der full-Dateien (7B/3B). Erst die Setup-Zelle ausfuehren.")

Gemeinsame Anzeigen 7B/3B: 49
3B-Uebereinstimmung mit 7B pro Feld:


,feld,3b_gleich_7b,abweichungen
0,homeoffice,35/49,14
1,vertragsart,43/49,6
2,erfahrungslevel,34/49,15
3,gehalt_min_eur,28/49,21
4,gehalt_zeitraum,28/49,21
5,skills_top3,10/49,39


Abweichungen 3B vs. 7B gesamt: 116


,refnr,feld,wert_7b,wert_3b,hat_gold,wert_gold,3b_halluzination
0,10001-1002993453-S,skills_top3,"[FMECA, Datenanalyse, Datenmodellierung]","[Data Science, Mathematik, FMECA]",False,,
1,10001-1002928664-S,erfahrungslevel,mid,junior,False,,
2,10001-1002928664-S,skills_top3,"[Power BI, Microsoft 365, Dynamics 365]","[Power BI, Dataverse, Copilot]",False,,
3,10001-1003016654-S,homeoffice,nicht_genannt,nein,False,,
4,10001-1003016654-S,gehalt_min_eur,NaN,50000.0,False,,
...,...,...,...,...,...,...,...
111,13509-00002110865001-S,gehalt_zeitraum,None,jahr,True,NaN,True
112,13509-00002110865001-S,skills_top3,"[Python, Data Science, Machine Learning]","[Python, Java, C#]",True,Java|DevOps|CI/CD,False
113,12862-244829-S,homeoffice,ja,teilweise,True,ja,True
114,12862-244829-S,erfahrungslevel,junior,mid,True,junior,True


Wahrscheinliche 3B-Halluzinationen (7B == Gold, 3B != Gold):


,refnr,feld,wert_7b,wert_3b,hat_gold,wert_gold,3b_halluzination
22,15939-BB-633095-7878-7490-S,vertragsart,festanstellung,sonstiges,True,festanstellung,True
36,15939-BB-633455-7878-6343-S,gehalt_min_eur,NaN,50000.0,True,NaN,True
37,15939-BB-633455-7878-6343-S,gehalt_zeitraum,None,jahr,True,NaN,True
45,15939-BB-633457-7878-2175-S,gehalt_min_eur,NaN,50000.0,True,NaN,True
46,15939-BB-633457-7878-2175-S,gehalt_zeitraum,None,jahr,True,NaN,True
76,13999-k53401.30280-S,erfahrungslevel,senior,mid,True,senior,True
77,13999-k53401.30280-S,gehalt_min_eur,NaN,50000.0,True,NaN,True
78,13999-k53401.30280-S,gehalt_zeitraum,None,jahr,True,NaN,True
92,15939-BB-632493-7878-9058-S,erfahrungslevel,junior,nicht_genannt,True,junior,True
97,13635-7fbe73ac_JB5131141-S,homeoffice,remote,ja,True,remote,True


## Schema-Konformität des vollen 7B-Runs

Auf den ~60+ Anzeigen ohne Gold lässt sich keine Accuracy rechnen, aber die Schema-Konformität ist selbst ein Befund über die Pipeline-Stabilität bei größerer Korpus-Vielfalt.

In [4]:
import subprocess
import sys

if PRED_7B_FULL.exists():
    res = subprocess.run(
        [sys.executable, "../annotation/validate.py", "--validate-jsonl", str(PRED_7B_FULL)],
        capture_output=True, text=True,
    )
    print(res.stdout)
    if res.stderr.strip():
        print("STDERR:", res.stderr)
    print("Notiz: auffaelligste Verletzungstypen hier kurz festhalten -> _..._")
else:
    print("Noch keine predictions_7b_full.jsonl - erst den 7B-Run in IterationB.ipynb ausfuehren.")


JSONL Schema-Check: predictions_7b_full.jsonl
Geprüfte Zeilen: 49
JSON-Parse-Fails: 0
Keine Feld-Verletzungen.

Notiz: auffaelligste Verletzungstypen hier kurz festhalten -> _..._


## Drei 3B-Halluzinations-Klassen

Aus den Abweichungs-Tabellen oben **drei eigene** Klassen benennen — nicht generisch („weicht ab"), sondern *auf welche Weise systematisch*. Jede Klasse soll auf mehrere Anzeigen passen.

| Klasse (eigener Name) | 3B-Beispiele (refnr + Wert) | 7B / Gold sagt | Vermutete Ursache |
|---|---|---|---|
| _z. B. „erfindet konkrete Pflicht-Skills"_ | `_refnr_`: skills=_…_ | _…_ | _…_ |
| _…_ | _…_ | _…_ | _…_ |
| _…_ | _…_ | _…_ | _…_ |

**Implikation:** Würde ich 3B im Produktivszenario einsetzen? Wenn ja, mit welcher Sicherung — Validator (`validate.py`), Stichproben-Review oder Eingrenzung auf die stabilen Felder (z. B. `vertragsart`)? _…_